<a href="https://colab.research.google.com/github/sebastiangome/latam-agente/blob/main/EP2_Agente_LATAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agente de Cuenta LATAM Airlines — Evaluación 2: Agente Funcional (ISY0101, EP2)

**Evaluación Parcial N°2 — Desarrollo de un Agente Funcional | Ingeniería de Soluciones con IA (ISY0101), Duoc UC**

**Integrantes:** Sebastián Gomez · Rafael Pacheco

Continuación del EP1 ([LATAM_RAG_Agent.ipynb](LATAM_RAG_Agent.ipynb)). El pipeline RAG pasa a ser un **agente** construido con **LangChain v1 (create_agent, sobre LangGraph)** que:

- **consulta** la reserva, las políticas oficiales y su historial de atenciones (herramientas de consulta),
- **razona** con la regla oficial de vuelo adelantado (herramienta de razonamiento),
- **escribe** casos para un ejecutivo humano (herramienta de escritura),
- **recuerda** dentro de la conversación (memoria de corto plazo) y entre conversaciones (memoria de largo plazo con búsqueda semántica),
- **planifica** los pasos según prioridad y **replanifica** cuando una herramienta cambia la situación.

Se reutilizan del EP1: el dataset de reservas, los 5 resúmenes de políticas con su URL, el SYSTEM_PROMPT y el stack Groq + Gemini + FAISS.

> **Credenciales:** LLM_API_KEY (Groq) y GOOGLE_API_KEY (Gemini) como **Secrets de Colab** con *Notebook access* activado, o en un archivo .env local.

---

### Declaración de Uso Ético de IA
[COMPLETAR POR EL EQUIPO] Herramientas de IA usadas, para qué se usaron y cómo validó el equipo el resultado, según la normativa de Duoc UC (bibliotecas.duoc.cl/ia).


## MÓDULO 1 — Setup y modelos

In [ ]:
# Instalación de dependencias (Google Colab)
!pip install -q langchain langgraph langchain-groq langchain-google-genai langchain-community faiss-cpu python-dotenv

In [ ]:
import os, re, json, sqlite3, unicodedata
from datetime import datetime
import numpy as np
import pandas as pd

# Credenciales: Colab Secrets o archivo .env (igual que en el EP1)
try:
    from google.colab import userdata
    for _k in ("LLM_API_KEY", "GOOGLE_API_KEY"):
        if os.getenv(_k): continue
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            print(f"[AVISO] Falta el Secret '{_k}': créalo en Secrets y activa 'Notebook access'")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

print("LLM_API_KEY (Groq):", "OK" if os.getenv("LLM_API_KEY") else "FALTA")
print("GOOGLE_API_KEY (Gemini):", "OK" if os.getenv("GOOGLE_API_KEY") else "FALTA")

In [ ]:
from langchain_groq import ChatGroq
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.messages import HumanMessage

# LLM: se prueba cada modelo en orden y se usa el primero disponible.
# En el EP1 Groq dio de baja llama-3.3-70b durante el proyecto; por eso hay respaldo.
MODELOS_GROQ = ["openai/gpt-oss-120b", "openai/gpt-oss-20b", "qwen/qwen3.6-27b"]
llm = None
for _modelo in MODELOS_GROQ:
    try:
        _cand = ChatGroq(model=_modelo, api_key=os.getenv("LLM_API_KEY"), temperature=0.2, max_tokens=2000)
        _cand.invoke([HumanMessage(content="ok")])
        llm, modelo_llm = _cand, _modelo
        print(f"[OK] LLM: {_modelo}")
        break
    except Exception as e:
        print(f"[INFO] {_modelo} no disponible: {str(e)[:80]}")
if llm is None:
    raise RuntimeError("No hay modelos Groq disponibles. Revisa LLM_API_KEY.")

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001",
                                          google_api_key=os.getenv("GOOGLE_API_KEY"))
print("[OK] Embeddings: gemini-embedding-001")

## MÓDULO 2 — Datos del EP1: reservas (fuente interna) y políticas (fuente externa)

In [ ]:
# Reservas simuladas del EP1 (200 filas, semilla 42)
RUTA_CSV = "data/reservas_latam.csv"
if not os.path.exists(RUTA_CSV):
    RUTA_CSV = "https://raw.githubusercontent.com/sebastiangome/latam-agente/main/data/reservas_latam.csv"
df_reservas = pd.read_csv(RUTA_CSV)
RESERVAS = {fila["codigo_reserva"]: fila for fila in df_reservas.to_dict(orient="records")}
print(f"[OK] {len(RESERVAS)} reservas cargadas")
df_reservas.head(3)

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

# Los mismos 5 resúmenes de políticas oficiales del EP1, con su URL
POLITICAS = [
    {"id": "ext_cambios_01", "tema": "cambios",
     "fuente_url": "https://www.latamairlines.com/cl/es/centro-ayuda/preguntas/cambios/pasajes/cambiar-vuelo-fecha-pasaje",
     "texto": "Un pasajero puede cambiar la fecha o el vuelo de su pasaje siempre que las condiciones de su tarifa lo permitan. El cambio debe solicitarse antes de la salida del vuelo original; en algunos casos tambien es posible cambiarlo despues de iniciado el viaje si la tarifa lo autoriza. El proceso se realiza desde la seccion 'Mis Viajes', ingresando el codigo de reserva y el apellido del pasajero."},
    {"id": "ext_cambios_02", "tema": "cambios",
     "fuente_url": "https://www.latamairlines.com/cl/es/centro-ayuda/preguntas/problemas-vuelo/cambio-itinerario/vuelo-adelantado",
     "texto": "Si LATAM adelanta un vuelo 16 minutos o mas y el pasajero no esta conforme con el nuevo horario, puede cambiar la hora o fecha del vuelo sin costo, o solicitar la devolucion del pasaje, siempre que mantenga el mismo destino y cabina del vuelo original. El plazo para solicitar este cambio es de hasta 12 meses desde la fecha del vuelo original comprado."},
    {"id": "ext_equipaje_01", "tema": "equipaje",
     "fuente_url": "https://www.latamairlines.com/us/es/centro-ayuda/preguntas/equipaje",
     "texto": "La franquicia de equipaje facturado depende de la tarifa, cabina y ruta del pasaje. En general, la cabina Economy permite una cantidad de piezas menor que Premium Economy o Premium Business. Ademas del equipaje facturado, se permite equipaje de mano y un articulo personal, con limites de peso y tamano que tambien varian segun la tarifa contratada. Es importante verificar la franquicia especifica en el sitio oficial antes de viajar, ya que estas politicas pueden cambiar (por ejemplo, LATAM ha reducido en el pasado la cantidad de piezas gratuitas en algunos vuelos domesticos)."},
    {"id": "ext_millas_01", "tema": "millas",
     "fuente_url": "https://www.latamairlines.com/es/es/centro-ayuda/preguntas/latam-pass/millas/como-acumular",
     "texto": "Los pasajeros acumulan Millas LATAM Pass al volar con LATAM o con aerolineas asociadas, asi como en comercios asociados o mediante tarjetas de credito con convenio. Los pasajes que fueron pagados totalmente con millas no generan acumulacion de nuevas millas ni de Puntos Calificables."},
    {"id": "ext_millas_02", "tema": "millas",
     "fuente_url": "https://latampass.latam.com/es_cl/reglamento-2025/acumulacion-y-canje",
     "texto": "El canje de equipaje adicional con Millas LATAM Pass solo puede realizarse dentro del mismo flujo de canje del pasaje; no es posible canjear equipaje antes o despues de haber canjeado el pasaje asociado. La cantidad de millas necesarias para un canje depende del destino y la cabina seleccionada."},
]

indice_politicas = FAISS.from_documents(
    [Document(page_content=p["texto"], metadata={k: p[k] for k in ("id", "tema", "fuente_url")}) for p in POLITICAS],
    embeddings,
)
print(f"[OK] Índice FAISS de políticas: {indice_politicas.index.ntotal} vectores")

## MÓDULO 3 — Memoria

- **Corto plazo:** el historial de cada conversación lo guarda el *checkpointer* de LangGraph (`InMemorySaver`), identificado por `thread_id`. Así el agente entiende un "es ACF657" como respuesta a la pregunta anterior.
- **Largo plazo:** una base SQLite guarda cada atención (resumen + vector de embedding) y los casos derivados. Persiste entre conversaciones.
- **Recuperación semántica:** se filtra primero por código de reserva y luego se ordenan las atenciones por similitud coseno con la consulta actual. Nunca se mezcla la historia de dos pasajeros (mismo criterio de privacidad del EP1).

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memoria_corto_plazo = InMemorySaver()

# True: la demo parte con la memoria de largo plazo vacía (evita casos de ensayos anteriores).
# False: se conserva lo guardado en ejecuciones previas.
REINICIAR_MEMORIA = True
if REINICIAR_MEMORIA and os.path.exists("memoria_latam.db"):
    os.remove("memoria_latam.db")

db = sqlite3.connect("memoria_latam.db", check_same_thread=False)
db.executescript('''
CREATE TABLE IF NOT EXISTS atenciones (pnr TEXT, fecha TEXT, resumen TEXT, vector TEXT);
CREATE TABLE IF NOT EXISTS casos (id TEXT, pnr TEXT, fecha TEXT, tipo TEXT, prioridad TEXT, detalle TEXT, estado TEXT);
''')

def guardar_atencion(pnr, resumen):
    vector = embeddings.embed_query(resumen)
    db.execute("INSERT INTO atenciones VALUES (?, ?, ?, ?)",
               (pnr, datetime.now().isoformat(timespec="seconds"), resumen, json.dumps(vector)))
    db.commit()

def buscar_atenciones(pnr, consulta, k=2):
    filas = db.execute("SELECT fecha, resumen, vector FROM atenciones WHERE pnr = ?", (pnr,)).fetchall()
    if not filas:
        return []
    q = np.array(embeddings.embed_query(consulta))
    m = np.array([json.loads(v) for _, _, v in filas])
    sim = m @ q / (np.linalg.norm(m, axis=1) * np.linalg.norm(q))
    orden = np.argsort(-sim)[:k]
    return [{"fecha": filas[i][0], "resumen": filas[i][1], "similitud": round(float(sim[i]), 3)} for i in orden]

def casos_abiertos(pnr):
    filas = db.execute("SELECT id, tipo, prioridad FROM casos WHERE pnr = ? AND estado = 'abierto'", (pnr,)).fetchall()
    return [{"id": i, "tipo": t, "prioridad": p} for i, t, p in filas]

print("[OK] Memoria de corto plazo (checkpointer) y largo plazo (SQLite) listas")

## MÓDULO 4 — Herramientas del agente

| Tipo | Herramienta | Qué hace |
|---|---|---|
| Consulta | `consultar_reserva` | Búsqueda **exacta** del código de reserva (como en el EP1) |
| Consulta | `buscar_politicas` | RAG sobre las políticas, filtrando por tema, con URL de la fuente |
| Consulta | `recordar_historial` | Memoria de largo plazo: atenciones parecidas y casos abiertos del pasajero |
| Razonamiento | `evaluar_vuelo_adelantado` | Regla oficial: ≥ 16 min de adelanto y ≤ 12 meses → cambio sin costo o devolución |
| Escritura | `registrar_caso` | Crea un caso para un ejecutivo (el agente no ejecuta transacciones) |

La regla de vuelo adelantado se evalúa en código porque en el EP1 el LLM llegó a dar cifras que no estaban en la política recuperada ("2 piezas de hasta 32 kg").

In [ ]:
from langchain_core.tools import tool

PATRON_PNR = re.compile(r"\b[A-Z]{3}\d{3}\b")

@tool
def consultar_reserva(codigo_reserva: str) -> str:
    """Busca una reserva por su código exacto (3 letras + 3 números, ej. RAG276)."""
    fila = RESERVAS.get(codigo_reserva.strip().upper())
    if fila is None:
        return json.dumps({"encontrada": False, "codigo_reserva": codigo_reserva})
    return json.dumps({"encontrada": True, **{k: (int(v) if k == "millas_acumuladas" else v) for k, v in fila.items()}},
                      ensure_ascii=False)

@tool
def buscar_politicas(consulta: str, tema: str = "") -> str:
    """Recupera las políticas oficiales de LATAM más relevantes. tema: cambios, equipaje o millas."""
    filtro = {"tema": tema} if tema in ("cambios", "equipaje", "millas") else None
    docs = indice_politicas.similarity_search(consulta, k=2, filter=filtro)
    return json.dumps([{"texto": d.page_content, "fuente_url": d.metadata["fuente_url"]} for d in docs],
                      ensure_ascii=False)

@tool
def recordar_historial(codigo_reserva: str, consulta: str) -> str:
    """Recupera atenciones anteriores parecidas y casos abiertos del pasajero (memoria de largo plazo)."""
    pnr = codigo_reserva.strip().upper()
    if pnr not in RESERVAS:
        return json.dumps({"error": "reserva no verificada"})
    return json.dumps({"atenciones_previas": buscar_atenciones(pnr, consulta),
                       "casos_abiertos": casos_abiertos(pnr)}, ensure_ascii=False)

@tool
def evaluar_vuelo_adelantado(minutos_adelanto: int, meses_desde_vuelo: float = 0) -> str:
    """Aplica la regla oficial: si el vuelo se adelantó 16 minutos o más y la solicitud es dentro
    de 12 meses, el pasajero puede cambiar sin costo o pedir la devolución."""
    elegible = minutos_adelanto >= 16 and meses_desde_vuelo <= 12
    if minutos_adelanto < 16:
        motivo = f"El adelanto ({minutos_adelanto} min) es menor a 16 minutos: no aplica el cambio sin costo."
    elif meses_desde_vuelo > 12:
        motivo = "La solicitud está fuera del plazo de 12 meses."
    else:
        motivo = "Elegible: puede cambiar hora o fecha sin costo, o pedir devolución, con el mismo destino y cabina."
    return json.dumps({"elegible": elegible, "motivo": motivo, "fuente_url": POLITICAS[1]["fuente_url"]},
                      ensure_ascii=False)

@tool
def evalua_vuelo_adelantado(minutos_adelanto: int, meses_desde_vuelo: float = 0) -> str:
    """Alias compatible para la herramienta evaluar_vuelo_adelantado."""
    return evaluar_vuelo_adelantado.invoke({"minutos_adelanto": minutos_adelanto,
                                             "meses_desde_vuelo": meses_desde_vuelo})

@tool
def registrar_caso(tipo: str, prioridad: str, detalle: str, codigo_reserva: str = "") -> str:
    """Registra un caso para que lo gestione un ejecutivo humano. El código de reserva es opcional. tipo: vuelo_adelantado | transaccion | fuera_de_cobertura. prioridad: alta | media | baja."""
    n = db.execute("SELECT COUNT(*) FROM casos").fetchone()[0] + 1
    caso_id = f"CASO-{n:04d}"
    db.execute("INSERT INTO casos VALUES (?, ?, ?, ?, ?, ?, 'abierto')",
               (caso_id, codigo_reserva.upper(), datetime.now().isoformat(timespec="seconds"), tipo, prioridad, detalle))
    db.commit()
    return json.dumps({"caso_id": caso_id, "prioridad": prioridad, "estado": "abierto"})

HERRAMIENTAS = [consultar_reserva, buscar_politicas, recordar_historial, evaluar_vuelo_adelantado,
                evalua_vuelo_adelantado, registrar_caso]
print("[OK] Herramientas:", [t.name for t in HERRAMIENTAS])

## MÓDULO 5 — Planificación con prioridades

Antes de **cada** llamada al modelo, `planificar()` arma el plan del turno a partir de la consulta y de lo que ya devolvieron las herramientas, y lo agrega al prompt de sistema. Así el agente:

1. **Ordena** los temas por prioridad: vuelo adelantado y transacciones (alta) → cambios y equipaje (media) → millas (baja).
2. **Verifica primero** la reserva (privacidad) y, si no hay código, lo pide.
3. **Replanifica** cuando cambian las condiciones: reserva inexistente, vuelo elegible a cambio sin costo, o caso ya abierto en la memoria de largo plazo.

In [ ]:
TEMAS = {  # tema: (palabras clave, prioridad)
    "vuelo_adelantado": (r"adelant", 1),
    "transaccion": (r"cambiame|canjeame|cancela|reembols|devuelveme|hazme el cambio", 1),
    "cambios": (r"cambi|modific|otra fecha", 2),
    "equipaje": (r"equipaje|maleta|pieza|kg|kilo", 2),
    "millas": (r"milla|latam pass|acumul|canje", 3),
}

def normalizar(texto):
    texto = unicodedata.normalize("NFKD", texto.lower())
    return "".join(c for c in texto if not unicodedata.combining(c))

def planificar(consulta, resultados=None):
    """Devuelve el plan del turno. resultados = {herramienta: salida} ya obtenidos en este turno."""
    resultados = resultados or {}
    t = normalizar(consulta)
    temas = sorted([tm for tm, (patron, _) in TEMAS.items() if re.search(patron, t)], key=lambda tm: TEMAS[tm][1])
    if "vuelo_adelantado" in temas and "cambios" in temas:
        temas.remove("cambios")
    if "transaccion" in temas and "cambios" in temas:
        temas.remove("cambios")
    pnr_m = PATRON_PNR.search(consulta.upper())
    pnr = pnr_m.group(0) if pnr_m else None
    minutos = re.search(r"(\d+)\s*min", t)
    pasos, decisiones = [], []

    if not temas:
        pasos.append("No hay política para este tema: NO respondas con conocimiento propio. registrar_caso(tipo='fuera_de_cobertura', prioridad='media') sin pedir el código de reserva, e informa al pasajero el ID del caso.")
        return {"temas": ["fuera_de_cobertura"], "pnr": pnr, "pasos": pasos, "decisiones": decisiones}

    # 1) Identidad / privacidad
    reserva = resultados.get("consultar_reserva")
    if pnr is None:
        pasos.append("Sin código de reserva: responde solo la política general y PIDE el código.")
    elif reserva is not None and not reserva.get("encontrada"):
        decisiones.append(f"Replanificación: la reserva {pnr} no existe → no entregar datos personales.")
        pasos.append("Pide al pasajero revisar el código. No uses recordar_historial ni registrar_caso.")
    else:
        pasos.append(f"consultar_reserva('{pnr}')")
        pasos.append(f"recordar_historial('{pnr}', consulta)")

    # 2) Temas en orden de prioridad
    historial = resultados.get("recordar_historial") or {}
    caso_previo = [c for c in historial.get("casos_abiertos", []) if c["tipo"] == "vuelo_adelantado"]
    for tema in temas:
        if tema == "vuelo_adelantado":
            if caso_previo:
                decisiones.append(f"Memoria de largo plazo: ya existe {caso_previo[0]['id']} → informar estado, no duplicar.")
                continue
            pasos.append("buscar_politicas(consulta, tema='cambios')")
            if minutos:
                pasos.append(f"evaluar_vuelo_adelantado(minutos_adelanto={minutos.group(1)})")
            else:
                pasos.append("Pide cuántos minutos se adelantó el vuelo.")
            evaluacion = resultados.get("evaluar_vuelo_adelantado")
            if evaluacion and evaluacion.get("elegible"):
                decisiones.append("Replanificación: vuelo elegible → registrar caso de prioridad ALTA.")
                pasos.append("registrar_caso(tipo='vuelo_adelantado', prioridad='alta')")
        elif tema == "transaccion":
            pasos.append("El agente NO ejecuta transacciones: registrar_caso(tipo='transaccion', prioridad='alta')")
        else:
            pasos.append(f"buscar_politicas(consulta, tema='{tema}')")
    pasos.append("Responde citando la fuente_url. No des cifras que no vengan de una herramienta.")
    return {"temas": temas, "pnr": pnr, "pasos": pasos, "decisiones": decisiones}

planificar("Reserva CKJ320: me adelantaron el vuelo 40 minutos y quiero saber cómo acumulo millas")

In [ ]:
# Pruebas del planificador (sin LLM)
p = planificar("¿Puedo cambiar mi vuelo?")
assert p["pnr"] is None and "PIDE el código" in p["pasos"][0]

p = planificar("Reserva RAG276: ¿cómo acumulo millas? Me adelantaron el vuelo 30 minutos")
assert p["temas"] == ["vuelo_adelantado", "millas"]          # prioridad alta antes que baja

p = planificar("Reserva ZZZ999, ¿equipaje?", {"consultar_reserva": {"encontrada": False}})
assert any("no existe" in d for d in p["decisiones"])         # replanifica: reserva inexistente

p = planificar("Reserva CKJ320, adelantaron 40 minutos", {"evaluar_vuelo_adelantado": {"elegible": True}})
assert any("registrar_caso" in s for s in p["pasos"])         # replanifica: vuelo elegible

p = planificar("Reserva QVA863, adelantaron 10 minutos", {"evaluar_vuelo_adelantado": {"elegible": False}})
assert not any("registrar_caso" in s for s in p["pasos"])     # no aplica: sin caso

p = planificar("¿Puedo viajar con mi perro?")
assert p["temas"] == ["fuera_de_cobertura"]

assert json.loads(evaluar_vuelo_adelantado.invoke({"minutos_adelanto": 15}))["elegible"] is False
assert json.loads(evaluar_vuelo_adelantado.invoke({"minutos_adelanto": 16}))["elegible"] is True
assert json.loads(evalua_vuelo_adelantado.invoke({"minutos_adelanto": 16}))["elegible"] is True
assert json.loads(consultar_reserva.invoke({"codigo_reserva": "ZZZ999"}))["encontrada"] is False
print("[OK] 10 pruebas del planificador y las herramientas aprobadas")

## MÓDULO 6 — Agente (`create_agent` de LangChain v1)

El agente es el bucle ReAct de LangGraph: el modelo decide qué herramienta llamar, ve el resultado y vuelve a decidir. Un *middleware* (`prompt_con_plan`) recalcula el plan antes de cada llamada al modelo y lo agrega al `SYSTEM_PROMPT` del EP1.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt
from langchain_core.messages import HumanMessage, ToolMessage

SYSTEM_PROMPT = """Eres un asistente de atencion al pasajero para LATAM Airlines.
Tu funcion es responder consultas sobre cambios de vuelo, equipaje y Millas LATAM Pass.

Reglas:
- Usa SOLO la informacion que entregan las herramientas (reserva del pasajero y politicas
  oficiales). No inventes datos ni reglas.
- Usa los nombres de herramientas exactamente como están definidos, especialmente evaluar_vuelo_adelantado.
- Si el pasajero indica los minutos de adelanto, evalúa primero la regla con evaluar_vuelo_adelantado; sigue el plan y registra el caso solo si la evaluación lo indica.
- No afirmes costos, cargos, piezas, kilos ni condiciones de una tarifa que no estén en la
  política recuperada: si dependen de la tarifa, dilo así y remite a latamairlines.com.
- Si la consulta depende de un dato que no tienes, pide ese dato al pasajero en vez de asumirlo.
- Las politicas de LATAM pueden cambiar: recomienda verificar la vigencia en latamairlines.com.
- No ejecutas transacciones reales (cambios, canjes); solo informas, orientas y registras casos.
- Responde en español, en un tono claro y cercano para un pasajero."""

def turno_actual(mensajes):
    """Consulta del pasajero (une la anterior si la actual solo trae un dato) y resultados del turno."""
    humanos = [i for i, m in enumerate(mensajes) if m.type == "human"]
    consulta = mensajes[humanos[-1]].content
    solo_un_dato = len(consulta.split()) <= 4 and planificar(consulta)["temas"] == ["fuera_de_cobertura"]
    if len(humanos) > 1 and solo_un_dato:   # continuidad: "es ACF657" responde a la pregunta anterior
        consulta = mensajes[humanos[-2]].content + " " + consulta
    resultados = {}
    for m in mensajes[humanos[-1]:]:
        if isinstance(m, ToolMessage):
            try:
                nombre = "evaluar_vuelo_adelantado" if m.name == "evalua_vuelo_adelantado" else m.name
                resultados[nombre] = json.loads(m.content)
            except json.JSONDecodeError:
                pass
    return consulta, resultados

@dynamic_prompt
def prompt_con_plan(request):
    consulta, resultados = turno_actual(request.state["messages"])
    plan = planificar(consulta, resultados)
    texto_plan = "\n".join(f"{i}. {p}" for i, p in enumerate(plan["pasos"], 1))
    decisiones = "\n".join(f"- {d}" for d in plan["decisiones"]) or "- ninguna"
    return f"{SYSTEM_PROMPT}\n\nPLAN (ordenado por prioridad, temas: {plan['temas']}):\n{texto_plan}\n\nDECISIONES:\n{decisiones}"

agente = create_agent(model=llm, tools=HERRAMIENTAS, middleware=[prompt_con_plan],
                      checkpointer=memoria_corto_plazo)

def atender(consulta, sesion):
    """Un turno de atención. Muestra plan, herramientas usadas y respuesta, y guarda la atención."""
    estado = agente.invoke({"messages": [HumanMessage(content=consulta)]},
                           config={"configurable": {"thread_id": sesion}})
    mensajes = estado["messages"]
    c, resultados = turno_actual(mensajes)
    plan = planificar(c, resultados)
    usadas = [m.name for m in mensajes[[i for i, m in enumerate(mensajes) if m.type == "human"][-1]:]
              if isinstance(m, ToolMessage)]
    respuesta = mensajes[-1].content
    print(f"PASAJERO: {consulta}\nTEMAS: {plan['temas']}")
    for d in plan["decisiones"]:
        print("DECISIÓN:", d)
    print("HERRAMIENTAS:", " → ".join(usadas) or "ninguna")
    print("\nAGENTE:", respuesta)
    reserva = resultados.get("consultar_reserva", {})
    if reserva.get("encontrada"):   # memoria de largo plazo solo para reservas verificadas
        guardar_atencion(reserva["codigo_reserva"], f"Consulta: {c[:200]} | Temas: {plan['temas']} | "
                                                    f"Respuesta: {respuesta[:200]}")
    return respuesta

## MÓDULO 7 — Demostración: toma de decisiones ante condiciones cambiantes (IE7)

### Escenario 1 — Consulta con reserva: consulta + RAG por tema, sin inventar cifras

In [ ]:
atender("Tengo la reserva RAG276, ¿cuánto equipaje puedo llevar?", sesion="s1");

### Escenario 2 — Falta el código y luego llega (memoria de corto plazo)

In [ ]:
atender("¿Puedo cambiar mi vuelo de mañana?", sesion="s2");

In [ ]:
atender("es ACF657", sesion="s2");

### Escenario 3 — Misma pregunta, condición distinta: 10 vs 40 minutos de adelanto

In [ ]:
atender("Reserva QVA863: me adelantaron el vuelo 10 minutos, ¿lo puedo cambiar gratis?", sesion="s3");

In [ ]:
atender("Reserva CKJ320: LATAM adelantó 40 minutos el vuelo de mañana, ¿lo puedo cambiar gratis?", sesion="s4");

### Escenario 4 — Reserva inexistente (privacidad)

In [ ]:
atender("Mi reserva es ZZZ999, ¿cuánto equipaje tengo?", sesion="s5");

### Escenario 5 — Consulta fuera de cobertura: no inventa, deriva

In [ ]:
atender("¿Puedo viajar con mi perro en la cabina?", sesion="s6");

### Escenario 6 — Nueva conversación, mismo pasajero (memoria de largo plazo)
Otra `sesion`: el agente no tiene el historial de chat, pero recupera el caso del escenario 3 desde SQLite y no lo duplica.

In [ ]:
atender("Hola, soy de la reserva CKJ320, ¿qué pasó con lo del vuelo adelantado?", sesion="s7");

### Lo que el agente escribió en la memoria de largo plazo

In [ ]:
display(pd.read_sql("SELECT * FROM casos", db))
display(pd.read_sql("SELECT pnr, fecha, substr(resumen, 1, 120) AS resumen FROM atenciones", db))